# Stage 3 — Architecture

**Goal:** understand the transformer well enough to have written it, and *prove*
the understanding is correct rather than assuming it.

### Why two implementations

`src/tinyllm/model_scratch.py` implements Llama from scratch — RMSNorm, RoPE,
GQA, SwiGLU, weight tying. `transformers.LlamaForCausalLM` implements the same
thing. Stage 4 trains HuggingFace's; this notebook proves ours computes the
identical function.

That split is deliberate:

- The from-scratch version is how you actually learn RoPE and GQA. Reading 200
  lines you can run beats reading a paper.
- HuggingFace's version has the tensor names `convert_hf_to_gguf.py` expects, so
  training it means stage 8 carries no conversion risk.
- The equality proof is what makes the from-scratch version trustworthy as a
  reference rather than a plausible-looking sketch.

An *almost* correct transformer is the worst outcome: it trains, the loss falls,
and the bug surfaces as "the model just isn't very good" after the GPU time is
already spent. RoPE convention errors and GQA head-mapping errors both fail
exactly that way.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it HF_TOKEN
# and enable notebook access. Never paste a token into a cell.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception as e:
    print(f"No HF_TOKEN yet ({e}). Needed from notebook 04 onward.")

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

## 3.1 — Where the parameters go

Before any code, the arithmetic. `config.param_breakdown()` computes this from
the shapes alone; later we check the real model agrees.

In [ ]:
pb = model_cfg.param_breakdown()
total = pb["total"]

print(f"{'component':<18} {'params':>12}   share")
print("-" * 44)
for k in ("embedding", "attention", "mlp", "layernorms", "lm_head"):
    print(f"{k:<18} {pb[k]:>12,}   {pb[k] / total:>6.1%}")
print("-" * 44)
print(f"{'total':<18} {total:>12,}")
print(f"{'non-embedding':<18} {pb['non_embedding']:>12,}")

print(f"\nper layer: {pb['per_layer']:,} x {model_cfg.num_hidden_layers} layers")
print(f"tied embeddings save {pb['embedding']:,} params "
      f"({pb['embedding'] / (total + pb['embedding']):.0%} of an untied model)")

The MLP holds ~60% of the parameters — that ratio is typical, and it's why SwiGLU's
`intermediate_size` is the first knob to reach for when resizing a model.

## 3.2 — The four Llama ingredients

### RMSNorm
LayerNorm without mean subtraction: just scale by root-mean-square. Cheaper, and
empirically as good. The subtlety is the dtype — the variance of an fp16
activation can lose all its precision, so normalization is computed in fp32 and
cast back. Getting this wrong is a classic silent-divergence bug.

### RoPE
Position is encoded by *rotating* q and k in 2D subspaces rather than adding
anything to the residual stream. Because a rotation by *m* composed with a
rotation by *−n* is a rotation by *m−n*, attention scores end up depending only
on **relative** distance.

The trap: HuggingFace pairs dimension *i* with *i + d/2*, not *i* with *i+1*. The
conventions are equivalent up to a permutation, but **not interchangeable for a
given set of weights**. Matching HF here is exactly what makes the parity check
pass.

### GQA
6 query heads share 2 key/value heads. `repeat_kv` is a view-and-reshape, so it
costs no FLOPs — the saving is in the **cache**. Only 2 heads' worth of K and V
are ever stored, cutting KV cache memory by 67%.

### SwiGLU
A gated MLP: `down(silu(gate(x)) * up(x))`. Three matrices instead of two, so the
hidden width is ~2.67× rather than 4× for the same parameter count.

In [ ]:
import torch
from tinyllm.model_scratch import RMSNorm, RotaryEmbedding, rotate_half, repeat_kv

# --- RMSNorm: scale-invariant, and unlike LayerNorm it does not re-center.
x = torch.randn(2, 4, model_cfg.hidden_size) * 5 + 3   # shifted and scaled
out = RMSNorm(model_cfg.hidden_size)(x)
print(f"RMSNorm  in  rms={x.pow(2).mean(-1).sqrt().mean():.3f}  mean={x.mean():+.3f}")
print(f"         out rms={out.pow(2).mean(-1).sqrt().mean():.3f}  mean={out.mean():+.3f}")
print("         (mean is NOT driven to 0 -- that is the difference from LayerNorm)")

# --- RoPE: the rotation only depends on relative distance.
rope = RotaryEmbedding(model_cfg.head_dim, model_cfg.max_position_embeddings)
cos, sin = rope(seq_len=8)
q = torch.randn(1, 1, 8, model_cfg.head_dim)
q_rot = q * cos.unsqueeze(0).unsqueeze(0) + rotate_half(q) * sin.unsqueeze(0).unsqueeze(0)

print(f"\nRoPE cos/sin tables: {tuple(cos.shape)}")
print("dot product of rotated q at pairs of positions with equal separation:")
for (a, b) in [(0, 1), (3, 4), (6, 7), (0, 4), (2, 6)]:
    d = torch.dot(q_rot[0, 0, a], q_rot[0, 0, b]).item()
    print(f"  pos {a}<->{b} (distance {b - a}): {d:+.4f}")
print("  -> equal distances give equal dot products; only the gap matters.")

# --- GQA: expansion is free, the saving is in what gets cached.
kv = torch.randn(1, model_cfg.num_key_value_heads, 8, model_cfg.head_dim)
expanded = repeat_kv(kv, model_cfg.n_rep)
print(f"\nGQA  cached K/V {tuple(kv.shape)} -> expanded {tuple(expanded.shape)}")
print(f"     KV cache is {1 - model_cfg.kv_dim / model_cfg.hidden_size:.0%} smaller "
      f"than with {model_cfg.num_attention_heads} full KV heads")

## 3.3 — The stage 3 gate: prove the two models are the same function

Four checks. The fourth is the one that earns its keep.

In [ ]:
from tinyllm.parity import run_all

results = run_all()

**Why check 4 matters most.** Logit and loss parity both run a single forward pass
starting at position 0, so they never exercise a non-zero RoPE offset. A KV-cache
bug — stale positions, re-rotating cached keys, an off-by-one in the offset —
leaves *training completely unaffected* and only corrupts inference. It survives
every other test here, and it would show up much later as "the GGUF is worse than
the checkpoint".

Comparing cached against uncached greedy generation is what pins it down.

## 3.4 — Predict throughput

We now have a FLOPs estimate. Stage 4 measures the real thing, and the gap between
prediction and measurement is Model FLOPs Utilization — the fraction of the GPU
you're actually using.

In [ ]:
fpt = model_cfg.flops_per_token()
total_flops = fpt * train_cfg.total_tokens
T4_PEAK = 65e12   # fp16 tensor-core peak

print(f"  fwd+bwd FLOPs/token     {fpt:.3e}")
print(f"  training tokens         {train_cfg.total_tokens:,}")
print(f"  total training FLOPs    {total_flops:.3e}")
print()
for mfu in (0.10, 0.20, 0.35, 0.50):
    secs = total_flops / (T4_PEAK * mfu)
    print(f"  at {mfu:.0%} MFU -> {secs / 60:5.1f} min")

print("\nSmall models get poor MFU: the kernels are too small to saturate the GPU,")
print("so launch overhead and memory bandwidth dominate. 15-30% is a realistic")
print("expectation here, and the measured number in stage 4 is the real answer.")

## 3.5 — Sanity: an untrained model should be maximally confused

A randomly initialised model has no information, so its loss should sit at
`ln(vocab_size)` — the entropy of a uniform distribution over the vocabulary.
Meaningfully lower means something leaked; meaningfully higher means the
initialization is broken.

In [ ]:
import math
from tinyllm.train import build_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(model_cfg, device)

ids = torch.randint(0, model_cfg.vocab_size, (4, 128), device=device)
with torch.no_grad():
    loss = model(input_ids=ids, labels=ids).loss.item()

expected = math.log(model_cfg.vocab_size)
print(f"  untrained loss   {loss:.4f}")
print(f"  ln(vocab_size)   {expected:.4f}")
print(f"  ratio            {loss / expected:.4f}")
print(f"  implied perplexity {math.exp(loss):,.0f} (vocab is {model_cfg.vocab_size:,})")
assert abs(loss - expected) < 0.5, "untrained loss is not near ln(V) -- check init"
print("\nThe model starts out guessing uniformly, as it should.")

del model
torch.cuda.empty_cache()

## Stage 3 gate

- [x] Parameter counts agree between both models and `config.param_breakdown()`
- [x] Logits match to < 1e-4
- [x] Loss matches, so both use the same label-shift convention
- [x] Cached generation == uncached generation (the RoPE-offset check)
- [x] Untrained loss ≈ ln(V)

The from-scratch implementation is now a *verified* reference for what gets
trained. Read `src/tinyllm/model_scratch.py` — it's the whole architecture in
about 200 lines, and you now know it's correct.

**Next:** `04_pretrain.ipynb`